In [8]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import IpBlocked, NoTranscriptFound, RequestBlocked, TranscriptsDisabled, VideoUnavailable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import os


Step 1a - indexing (Documents Ingestion) - YouTube Transcript

In [9]:
video_id = "0Rwb4Xmlcwc"  # Replace with the actual YouTube video

if video_id == "YOUR_VIDEO_ID":
    raise ValueError("Replace video_id with the YouTube video ID before running this cell.")

try:
    # Fetch the preferred English transcript.
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=("en",))

    # Flatten the transcript snippets to plain text.
    transcript_text = " ".join(snippet.text for snippet in transcript)
    print("Transcript fetched successfully.", transcript_text[:500])
except (NoTranscriptFound, TranscriptsDisabled):
    raise RuntimeError(f"No transcript is available for video {video_id!r}.")
except VideoUnavailable:
    raise RuntimeError(f"Video {video_id!r} is unavailable.")
except (IpBlocked, RequestBlocked) as exc:
    raise RuntimeError(
        "YouTube blocked this request. Run the notebook from a different network "
        "or configure a supported proxy, then retry."
    ) from exc


Transcript fetched successfully. backend engineering is a very wide scope and when I say backend engineering it is much more than building a set of crud apis the way I think backend engineering is about building reliable scalable fall tolerant and maintainable code bases and efficient systems and if one were to start today to learn back in development there are a lot of resources and at least 1,00 resources but how do you decide what to learn how do you prioritize how do you see the big picture when and how all these different 


Step 1b - indexing (Documents Ingestion) - YouTube Transcript

In [10]:
splitters = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitters.create_documents([transcript_text])

In [11]:
len(chunks)  # Check how many chunks were created

41

In [12]:
chunks[0].page_content[:500]  # Display the first 500 characters of the first chunk

'backend engineering is a very wide scope and when I say backend engineering it is much more than building a set of crud apis the way I think backend engineering is about building reliable scalable fall tolerant and maintainable code bases and efficient systems and if one were to start today to learn back in development there are a lot of resources and at least 1,00 resources but how do you decide what to learn how do you prioritize how do you see the big picture when and how all these different '

Step 1c & 1d - Indexing(Embedding creation and Vector Store Creation) - YouTube Transcript

In [13]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=30,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

vector_store = FAISS.from_documents(chunks, embedding_model)


In [14]:
vector_store.index_to_docstore_id

{0: 'bc586b4b-86e9-4a39-94c4-fc74ecdf7f20',
 1: '5059d5c0-ebbe-46ef-96c2-06bbd938a048',
 2: '7689285e-e9fb-4b54-8460-df55ebd660b3',
 3: 'd4e2abb7-756d-42f8-bf8c-d4515e889383',
 4: '4c78668a-e35d-4f9f-a58c-0bf743fb19ae',
 5: '04d4500a-cf1a-4ef5-9459-680f9edf0a55',
 6: '26b50906-6db0-4344-bb48-ae91ea8bbe91',
 7: '38cf723c-18ed-48a5-9ad1-ccf4c64155cd',
 8: '157cc013-2906-4927-94a9-db66ea216073',
 9: 'c2526dbd-4233-4807-83f5-cb7dfef54c9f',
 10: 'b666f210-3479-40b6-bf3a-e8343d4e737b',
 11: '44589cb8-2421-4e94-83a5-7a774f153361',
 12: 'afbe5322-f772-4d71-a729-f7523524512e',
 13: '81f9d19a-449d-4951-8aeb-60318a65fd0f',
 14: 'dbf1062c-12a2-4b40-87d4-16257b5fcabb',
 15: '53e72cd3-feba-46eb-bafe-025ceeae4097',
 16: 'e91657a8-059f-4e23-b0c7-932c13e57799',
 17: '8c547614-75e6-40d2-ad84-571946bc89ed',
 18: 'e55599ca-ea5e-4737-86c3-7c000ecfa100',
 19: 'dfc66066-c6a2-428d-823a-52fcefdee113',
 20: '1a34616c-9e8b-491d-9044-868306548151',
 21: 'efb58685-edb1-4b80-ad75-082f52247a6b',
 22: 'eae67194-4361-

In [15]:
vector_store.get_by_ids(["77b82958-c5f2-449f-8142-ecac2ffa9de5"]) # Retrieve the first three documents by their IDs

[]

Step 2 - Retrieval and Question Answering - YouTube Transcript

In [16]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 4})

In [17]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x75c2e7656e40>, search_type='mmr', search_kwargs={'k': 4})

In [18]:
retriever.invoke("What is backend from first principles?")  # Ask a question to the retriever

[Document(id='bc586b4b-86e9-4a39-94c4-fc74ecdf7f20', metadata={}, page_content='backend engineering is a very wide scope and when I say backend engineering it is much more than building a set of crud apis the way I think backend engineering is about building reliable scalable fall tolerant and maintainable code bases and efficient systems and if one were to start today to learn back in development there are a lot of resources and at least 1,00 resources but how do you decide what to learn how do you prioritize how do you see the big picture when and how all these different concepts come together and this is the reason it takes people years to get their head around a lot of these Concepts and principles because people primarily start with a limited scope of training uh whether it is from a college or a boot camp or a simple cop course and they eventually build onp top of that with trial and error and with help of other developers over time now I am a backend engineer and I have faced th

Step 3 - Augmented Generation - YouTube Transcript

In [19]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant that answers questions based on the context provided.
    If the answer is not contained within the context, say "I don't know."
    Context: {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [20]:
question = "what topics are covered in the video?"
retriever_docs = retriever.invoke(question)

In [21]:
for doc in retriever_docs:  # Display the retrieved documents
    print(f"Document: {doc.page_content[:100]}...")

Document: we'll see how ORS work whether to use an orm what are the tradeoffs and we'll look at what are datab...
Document: side caching and server side caching the different caching strategies for example cash aside right t...
Document: and efficient ensuring middleware is applied in the correct order or how how middleware order can af...
Document: switch to a different language let's say you were working with rubyan rails for years and one day yo...


In [22]:
context = "\n\n".join([doc.page_content for doc in retriever_docs])

In [23]:
print("Context:\n", context)  # Display the first 500 characters of the context

Context:
 we'll see how ORS work whether to use an orm what are the tradeoffs and we'll look at what are database migrations that we'll move on to business logic layer which is also called BL what is the role of it what are the different layers of a request cycle for example we have validation layer we have routing we have middlewares and we have handl and controls which all of them fall under presentation layer because they they deal with users data whether it is accepting users data or sending a user data so those are part of a presentation layer after that we have business logic layer which is the middle one which deals with our Core Business logic and after that we have the data access layer which deals with databases perform squaring or inserts or deletions and business logic layer uses the data access layer behind the scenes we'll look at different design principles like separation of concerns single responsibility open close dependency inversion what are the components of a busi

In [24]:
final_prompt = prompt.invoke({"context": context, "question": question})

Step 5 - Generation of Answer - YouTube Transcript

In [25]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [26]:
answer = model.invoke(final_prompt)

In [27]:
print(answer.content)

The video covers the following topics:

1. ORS and ORM tradeoffs
2. Database migrations
3. Business Logic Layer (BL) and its role
4. Layers of a request cycle, including:
   - Validation layer
   - Routing
   - Middlewares
   - Handlers and controllers (presentation layer)
5. Business logic layer and data access layer interactions
6. Design principles like separation of concerns, single responsibility, open/closed principle, and dependency inversion
7. Caching strategies:
   - Client-side and server-side caching
   - Different caching strategies (cache aside, write through, write behind, write back, read through)
   - Cache eviction strategies (LRU, LFU, TTL, FIFO)
   - Cash invalidation techniques (manual, Time To Live, event-based)
   - Levels of caching (level one: in-memory; level two: network distributed; hierarchical caching)
8. Caching for web apps (static assets and API responses)
9. Database caching (query caching)
10. Middleware application and its impact on performance and s

In [28]:
from langchain_classic.retrievers import ContextualCompressionRetriever  # noqa: F401
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


In [29]:
compressor = LLMChainExtractor.from_llm(llm=model)


In [30]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)


In [31]:
compressed_docs = compression_retriever.invoke(question)
compressed_context = "\n\n".join(doc.page_content for doc in compressed_docs)
compressed_prompt = prompt.invoke(
    {"context": compressed_context, "question": question}
)
answer = model.invoke(compressed_prompt)

In [32]:
print(answer.content)

The topics covered in the video include:

1. ORMs (Object-Relational Mapping) and their tradeoffs.
2. Database migrations.
3. The Business Logic Layer (BL) and its role.
4. Different layers of a request cycle, including:
   - Validation layer
   - Routing
   - Middlewares
   - Handlers and controllers (presentation layer)
5. Data Access Layer and its interaction with the Business Logic Layer.
6. Design principles such as separation of concerns, single responsibility, open/close principle, and dependency inversion.
7. Caching strategies, including:
   - Side caching (client-side and server-side)
   - Cache aside, write-through, write-behind, or write-back, read-through strategies.
   - Eviction strategies like LRU, LFU, TTL, and FIFO.
   - Invalidation techniques (manual, TTL, event-based).
   - Levels of caching (Level 1: in-memory, Level 2: network distributed, hierarchical caching).
8. Caching for web applications, including caching static assets and API responses using headers.
9. Q

Building youtbube chat boat using chain


In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [34]:
parser = StrOutputParser()

In [38]:
parallel_chain  = RunnableParallel(
    {
        "context": retriever | RunnableLambda(lambda docs: "\n\n".join(doc.page_content for doc in docs)),
        "question": RunnablePassthrough(),
    }
)

In [39]:
main_chain = parallel_chain | prompt | model | parser

In [40]:
result = main_chain.invoke(question)

In [41]:
print(result)

The video covers the following topics:

1. ORS (Object-Relational Mapping) and the trade-offs of using an ORM.
2. Database migrations.
3. Business Logic Layer (BL) and its role.
4. Different layers of a request cycle, including:
   - Validation layer
   - Routing
   - Middlewares
   - Handlers and controllers (part of the presentation layer)
5. The separation of concerns and design principles (single responsibility, open/close, dependency inversion).
6. Caching strategies, including:
   - Client-side and server-side caching
   - Cache-aside, write-through, write-behind, and read-through strategies
   - Cache eviction strategies (LRU, LFU, TTL, FIFO)
   - Need for cache invalidation (manual, TTL, event-based)
   - Levels of caching (level one – in-memory, level two – network distributed, hierarchical caching)
7. Caching for web apps (static assets, API responses using headers).
8. Caching with databases (query caching).
9. Middleware application and its impact on performance and securit